# Qudit cluster chain 50 site DMRG

Created 05/03/2026

Calculate a qudit cluster chain MPS with 50 sites, $d=7$.

In [2]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [3]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS

In [4]:
from tenpy.networks.site import SpinSite, ClockSite
from tenpy.models.lattice import Chain
from tenpy.models.model import CouplingModel, NearestNeighborModel, MPOModel

In [26]:
class QuditClusterChain(CouplingModel):
    def __init__(self, model_params):
        # Read out/set default parameters
        name = "Cluster Ising model"
        L = model_params.get('L', 5)
        d = model_params.get('d', 7)
        bc_MPS = model_params.get('bc_MPS', 'finite')
        
        # sites
        site = ClockSite(d, conserve=None)

        # lattice
        bc = 'open'
        lat = Chain(L, site, bc=bc, bc_MPS=bc_MPS)

        # initialize CouplingModel
        CouplingModel.__init__(self, lat)

        for i in range(1, L-1):
            if i % 2 == 0:
                self.add_local_term(
                    -1,
                    [
                        ('Zhc', (i-1, 0)),
                        ('X', (i, 0)),
                        ('Z', (i+1, 0)),
                    ],
                    plus_hc=True
                )
            else:
                self.add_local_term(
                    -1,
                    [
                        ('Z', (i-1, 0)),
                        ('X', (i, 0)),
                        ('Zhc', (i+1, 0)),
                    ],
                    plus_hc=True
                )

        # initialize H_MPO
        MPOModel.__init__(self, lat, self.calc_H_MPO())

In [27]:
import h5py
from tenpy.tools import hdf5_io

In [28]:
d=7

In [29]:
dmrg_params = {
    "mixer": True,
    "max_E_err": 1e-10,
    "trunc_params": {
        "chi_max": 2*d,
        "svd_min": 1e-10
    },
    "max_sweeps": 10
}

In [30]:
model=QuditClusterChain({'d': d, 'L':50})

psi = MPS.from_desired_bond_dimension(
    model.lat.mps_sites(),
    d,
    bc=model.lat.bc_MPS
)
psi.canonical_form()

In [31]:
eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
e, psi = eng.run()

print(e)

-96.00000000000139


In [33]:
filename = r"../data/qudit_cluster_chain_d_7_L_50.h5"
with h5py.File(filename, 'w') as f:
    hdf5_io.save_to_hdf5(f, psi)